# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


## Agent Workflow

```text
                 User Query
                      │
                      ▼
               Check Intent
        ┌─────────────┼─────────────┐
        │             │             │
        ▼             ▼             ▼
  "calculate"   "keywords"    Otherwise
        │             │             │
        ▼             ▼             ▼
 Calculator Tool  Keyword Tool  General Response
        │             │             │
        └─────────────┼─────────────┘
                      ▼
                Return JSON
```

In [31]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        # Safe evaluation (no built-in functions allowed)
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception:
        return "Error in calculation"

In [32]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [34]:
# 🤖 AGENT FUNCTION
import logging

# basic logging setup
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def agent(query: str):
    query_lower = query.lower()
    logger.info("Received query: " + query)

    # route to calculator if query mentions calculate
    if "calculate" in query_lower:
        try:
            # extract the math expression after the word calculate
            expression = query_lower.split("calculate", 1)[1].strip()
            logger.info("Routing to calculator with: " + expression)
            result = calculator(expression)
            if result == "Error in calculation":
                logger.warning("Calculator returned an error")
                return {
                    "type": "error",
                    "result": "Could not evaluate the expression. Please check the format."
                }
            return {
                "type": "calculation",
                "result": result
            }
        except Exception as e:
            logger.error("Error in calculation routing: " + str(e))
            return {
                "type": "error",
                "result": "Something went wrong while processing the calculation."
            }

    # route to keyword extractor if query mentions keywords
    elif "keywords" in query_lower:
        try:
            # take text after from if present, else use whole query
            if "from" in query_lower:
                text = query.split("from", 1)[1].strip()
            else:
                text = query
            logger.info("Routing to keyword extractor")
            result = extract_keywords(text)
            return {
                "type": "keywords",
                "result": result
            }
        except Exception as e:
            logger.error("Error in keyword routing: " + str(e))
            return {
                "type": "error",
                "result": "Something went wrong while extracting keywords."
            }

    # fallback for all other queries - general response
    else:
        logger.info("No specific tool matched, going to general response")
        return {
            "type": "general",
            "result": "I received your query: '" + query + "'. This is a general response."
        }


## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [36]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

2026-07-11 22:11:57,711 - INFO - Received query: Calculate 20 + 5
2026-07-11 22:11:57,713 - INFO - Routing to calculator with: 20 + 5
2026-07-11 22:11:57,713 - INFO - Received query: Extract keywords from Artificial Intelligence is transforming industries
2026-07-11 22:11:57,713 - INFO - Routing to keyword extractor
2026-07-11 22:11:57,714 - INFO - Received query: What is machine learning?
2026-07-11 22:11:57,715 - INFO - No specific tool matched, going to general response


Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['transforming', 'industries', 'artificial', 'intelligence']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': "I received your query: 'What is machine learning?'. This is a general response."}
--------------------------------------------------


In [37]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

Enter query (type 'exit' to stop):  exit
